# Phase 4: Graph-LeJEPA Benchmarking

This notebook benchmarks Graph-LeJEPA (Graph Joint-Embedding Predictive Architecture with SIGReg) against baseline models for EV charging demand prediction.

**Model Overview**:
- Graph-LeJEPA combines Graph-JEPA's masked subgraph prediction with LeJEPA's SIGReg regularization
- Uses GNN encoder (GCN/GIN) for learning spatial relationships between charging stations
- SIGReg (Sigmoid Regularization) prevents embedding collapse without EMA teacher
- O(N) complexity through Cramer-Wold random projections

**Objectives**:
- Benchmark Graph-LeJEPA on Shenzhen (SZH) dataset
- Compare with baseline models (LSTM, ModernTCN, ConvTimeNet)
- Evaluate the benefit of graph-based spatial modeling

In [12]:
import os
import sys
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.spatial.distance import cdist

# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))

from api.utils import calculate_regression_metrics, random_seed, create_rnn_data

# Import Graph-LeJEPA components
from g_lejepa import GraphLeJEPA, GNNEncoder, SIGRegLoss

print("Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Imports successful
PyTorch version: 2.9.1+cu128
CUDA available: False


## Configuration

In [13]:
# Experiment configuration
CONFIG = {
    'city': 'SZH',  # Shenzhen dataset
    'feature': 'volume',
    'auxiliary': 'all',  # Use all auxiliary features
    
    # Sequence configuration
    'seq_len': 24,  # 24 hours lookback
    'pred_len': 1,  # 1 hour ahead prediction
    
    # Site selection
    'max_sites': 100,  # Limit sites for faster experimentation
    
    # Graph-LeJEPA specific
    'hidden_dim': 128,
    'embedding_dim': 64,
    'num_gnn_layers': 3,
    'gnn_type': 'gcn',  # 'gcn' or 'gin'
    'mask_ratio': 0.15,
    'lambda_reg': 1.0,  # SIGReg regularization strength
    'num_projections': 256,  # For SIGReg
    'dropout': 0.1,
    
    # Training configuration
    'pretrain_epochs': 30,  # Pre-training epochs for representation learning
    'finetune_epochs': 30,  # Fine-tuning epochs for prediction
    'batch_size': 32,
    'pretrain_lr': 1e-3,
    'finetune_lr': 1e-4,
    'weight_decay': 1e-5,
    
    # Cross-validation
    'train_ratio': 0.7,
    'valid_ratio': 0.15,
    'test_ratio': 0.15,
    
    # Baseline models for comparison
    'baseline_models': ['lstm', 'moderntcn', 'convtimenet'],
    'baseline_epochs': 50,
    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}

# Set random seeds
random_seed(CONFIG['seed'])

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  city: SZH
  feature: volume
  auxiliary: all
  seq_len: 24
  pred_len: 1
  max_sites: 100
  hidden_dim: 128
  embedding_dim: 64
  num_gnn_layers: 3
  gnn_type: gcn
  mask_ratio: 0.15
  lambda_reg: 1.0
  num_projections: 256
  dropout: 0.1
  pretrain_epochs: 30
  finetune_epochs: 30
  batch_size: 32
  pretrain_lr: 0.001
  finetune_lr: 0.0001
  weight_decay: 1e-05
  train_ratio: 0.7
  valid_ratio: 0.15
  test_ratio: 0.15
  baseline_models: ['lstm', 'moderntcn', 'convtimenet']
  baseline_epochs: 50
  device: cpu
  seed: 42


## Custom Dataset Loading

We create a custom data loader that properly handles the Shenzhen dataset format.

In [14]:
class EVGraphDataset:
    """
    Custom dataset loader for EV charging data with graph structure.
    Handles the specific format of the CHARGED dataset.
    """
    
    def __init__(self, data_path, feature='volume', max_sites=100, auxiliary='all'):
        self.data_path = data_path
        self.feature = feature
        self.max_sites = max_sites
        self.auxiliary = auxiliary
        
        # Load main feature data
        print(f"Loading {feature} data...")
        self.feat_df = pd.read_csv(f'{data_path}{feature}.csv', index_col=0)
        self.time = pd.to_datetime(self.feat_df.index)
        
        # Load sites info - handle different column names
        print("Loading site information...")
        sites_df = pd.read_csv(f'{data_path}sites.csv')
        
        # Determine the site ID column
        if 'site_id' in sites_df.columns:
            site_col = 'site_id'
        elif 'site' in sites_df.columns:
            site_col = 'site'
        else:
            # Use first column as index
            site_col = sites_df.columns[0]
        
        sites_df = sites_df.set_index(site_col)
        sites_df.index = sites_df.index.astype(str)
        
        # Select top sites by activity
        if len(sites_df) > max_sites:
            print(f"Selecting top {max_sites} sites by total_duration...")
            if 'total_duration' in sites_df.columns:
                selected_sites = sites_df.sort_values('total_duration', ascending=False).head(max_sites)
            else:
                selected_sites = sites_df.head(max_sites)
            selected_ids = selected_sites.index.tolist()
            self.feat_df = self.feat_df[selected_ids]
            sites_df = selected_sites
        
        self.sites_df = sites_df
        self.site_ids = self.feat_df.columns.tolist()
        self.n_sites = len(self.site_ids)
        
        # Extract lat/long for graph construction
        self.lat_long = sites_df.loc[self.site_ids, ['latitude', 'longitude']].values
        # Normalize lat/long
        self.lat_long_norm = np.zeros_like(self.lat_long)
        self.lat_long_norm[:, 0] = (self.lat_long[:, 0] + 90) / 180  # lat: [-90, 90] -> [0, 1]
        self.lat_long_norm[:, 1] = (self.lat_long[:, 1] + 180) / 360  # lon: [-180, 180] -> [0, 1]
        
        # Convert to numpy
        self.feat = self.feat_df.values
        
        # Load auxiliary features if requested
        self.extra_feat = None
        if auxiliary != 'None':
            self._load_auxiliary_features()
        
        print(f"Dataset loaded: {self.feat.shape[0]} timesteps, {self.n_sites} sites")
    
    def _load_auxiliary_features(self):
        """Load and normalize auxiliary features."""
        aux_list = []
        
        # Load price data
        try:
            e_price = pd.read_csv(f'{self.data_path}e_price.csv', index_col=0)
            e_price = e_price[self.site_ids].values
            e_price = MinMaxScaler().fit_transform(e_price)
            aux_list.append(e_price[:, :, np.newaxis])
            print("  Loaded electricity price")
        except Exception as e:
            print(f"  Could not load e_price: {e}")
        
        try:
            s_price = pd.read_csv(f'{self.data_path}s_price.csv', index_col=0)
            s_price = s_price[self.site_ids].values
            s_price = MinMaxScaler().fit_transform(s_price)
            aux_list.append(s_price[:, :, np.newaxis])
            print("  Loaded service price")
        except Exception as e:
            print(f"  Could not load s_price: {e}")
        
        # Load weather data
        try:
            weather = pd.read_csv(f'{self.data_path}weather.csv', index_col='time')
            weather_cols = ['temp', 'precip', 'visibility']
            weather = weather[[c for c in weather_cols if c in weather.columns]]
            
            # Normalize weather
            if 'temp' in weather.columns:
                weather['temp'] = (weather['temp'] + 5) / 45
            if 'precip' in weather.columns:
                weather['precip'] = weather['precip'] / 120
            if 'visibility' in weather.columns:
                weather['visibility'] = weather['visibility'] / 50
            
            # Repeat weather for all sites
            weather_data = np.repeat(weather.values[:, np.newaxis, :], self.n_sites, axis=1)
            aux_list.append(weather_data)
            print(f"  Loaded weather ({weather.shape[1]} features)")
        except Exception as e:
            print(f"  Could not load weather: {e}")
        
        if aux_list:
            self.extra_feat = np.concatenate(aux_list, axis=2)
            print(f"  Total auxiliary features: {self.extra_feat.shape[2]}")
    
    def split_data(self, train_ratio=0.7, valid_ratio=0.15):
        """Split data into train/valid/test sets."""
        n = len(self.feat)
        train_end = int(n * train_ratio)
        valid_end = int(n * (train_ratio + valid_ratio))
        
        # Split main features
        train_feat = self.feat[:train_end]
        valid_feat = self.feat[train_end:valid_end]
        test_feat = self.feat[valid_end:]
        
        # Normalize using training data statistics
        self.scaler = StandardScaler()
        self.train_feat = self.scaler.fit_transform(train_feat)
        self.valid_feat = self.scaler.transform(valid_feat)
        self.test_feat = self.scaler.transform(test_feat)
        
        # Split auxiliary features
        if self.extra_feat is not None:
            self.train_extra = self.extra_feat[:train_end]
            self.valid_extra = self.extra_feat[train_end:valid_end]
            self.test_extra = self.extra_feat[valid_end:]
        else:
            self.train_extra = self.valid_extra = self.test_extra = None
        
        print(f"Data split: train={len(self.train_feat)}, valid={len(self.valid_feat)}, test={len(self.test_feat)}")


class TimeSeriesDataset(Dataset):
    """PyTorch Dataset for time series with sliding window."""
    
    def __init__(self, feat, extra_feat, seq_len, pred_len, device):
        self.device = device
        self.seq_len = seq_len
        self.pred_len = pred_len
        
        # Create sliding window samples
        X, y = create_rnn_data(feat, seq_len, pred_len)
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
        if extra_feat is not None:
            X_extra, _ = create_rnn_data(extra_feat, seq_len, pred_len)
            self.X_extra = torch.FloatTensor(X_extra)
        else:
            self.X_extra = None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].T.to(self.device)  # [nodes, seq_len]
        y = self.y[idx].to(self.device)    # [nodes]
        
        if self.X_extra is not None:
            # extra: [seq_len, nodes, n_aux] -> [nodes, seq_len, n_aux]
            extra = self.X_extra[idx].permute(1, 0, 2).to(self.device)
        else:
            extra = torch.empty(0, device=self.device)
        
        return x, y, extra

## Load Dataset

In [15]:
# Load EV dataset
data_path = f'../data/{CONFIG["city"]}_remove_zero/'
print(f"Loading dataset from {data_path}...\n")

ev_dataset = EVGraphDataset(
    data_path=data_path,
    feature=CONFIG['feature'],
    max_sites=CONFIG['max_sites'],
    auxiliary=CONFIG['auxiliary']
)

# Split data
ev_dataset.split_data(
    train_ratio=CONFIG['train_ratio'],
    valid_ratio=CONFIG['valid_ratio']
)

# Create data loaders
device = torch.device(CONFIG['device'])

train_dataset = TimeSeriesDataset(
    ev_dataset.train_feat, ev_dataset.train_extra,
    CONFIG['seq_len'], CONFIG['pred_len'], device
)
valid_dataset = TimeSeriesDataset(
    ev_dataset.valid_feat, ev_dataset.valid_extra,
    CONFIG['seq_len'], CONFIG['pred_len'], device
)
test_dataset = TimeSeriesDataset(
    ev_dataset.test_feat, ev_dataset.test_extra,
    CONFIG['seq_len'], CONFIG['pred_len'], device
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

n_sites = ev_dataset.n_sites
n_features = 1 + (ev_dataset.extra_feat.shape[-1] if ev_dataset.extra_feat is not None else 0)

print(f"\nDataset summary:")
print(f"  Sites: {n_sites}")
print(f"  Features per site: {n_features}")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(valid_dataset)}")
print(f"  Test samples: {len(test_dataset)}")

Loading dataset from ../data/SZH_remove_zero/...

Loading volume data...
Loading site information...
Selecting top 100 sites by total_duration...
  Loaded electricity price
  Loaded service price
  Loaded weather (3 features)
  Total auxiliary features: 5
Dataset loaded: 4392 timesteps, 100 sites
Data split: train=3074, valid=659, test=659

Dataset summary:
  Sites: 100
  Features per site: 6
  Training samples: 3049
  Validation samples: 634
  Test samples: 634


## Build Spatial Graph from Site Locations

We construct a spatial graph where:
- Each node represents a charging station
- Edges connect stations within a certain distance threshold
- This captures spatial dependencies between nearby stations

In [16]:
def build_spatial_graph(lat_long, threshold_percentile=25):
    """
    Build a spatial graph based on geographic proximity.
    
    Args:
        lat_long: Normalized latitude/longitude coordinates [N, 2]
        threshold_percentile: Connect nodes within this percentile of distances
    
    Returns:
        edge_index: [2, E] tensor of edges
    """
    # Compute pairwise distances
    distances = cdist(lat_long, lat_long, metric='euclidean')
    
    # Set distance threshold based on percentile
    # (excluding self-loops which have distance 0)
    non_zero_distances = distances[distances > 0]
    threshold = np.percentile(non_zero_distances, threshold_percentile)
    
    # Create adjacency based on threshold
    adjacency = (distances <= threshold) & (distances > 0)
    
    # Add self-loops
    np.fill_diagonal(adjacency, True)
    
    # Convert to edge index format
    edge_index = np.array(np.where(adjacency))
    
    return torch.tensor(edge_index, dtype=torch.long)

# Build spatial graph
edge_index = build_spatial_graph(ev_dataset.lat_long_norm, threshold_percentile=25)
edge_index = edge_index.to(device)

print(f"Spatial Graph Statistics:")
print(f"  Nodes: {n_sites}")
print(f"  Edges: {edge_index.shape[1]}")
print(f"  Avg degree: {edge_index.shape[1] / n_sites:.2f}")

Spatial Graph Statistics:
  Nodes: 100
  Edges: 2576
  Avg degree: 25.76


## Graph-LeJEPA Prediction Model

We wrap Graph-LeJEPA with a prediction head for time series forecasting.

In [17]:
class GraphLeJEPAPredictor(nn.Module):
    """
    Graph-LeJEPA model adapted for EV charging demand prediction.
    
    Architecture:
    1. Temporal encoding: Process time series per node
    2. Spatial encoding: GNN to capture inter-station dependencies
    3. Prediction head: Project to forecast horizon
    """
    
    def __init__(
        self,
        num_nodes: int,
        seq_len: int,
        pred_len: int,
        n_features: int,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_gnn_layers: int = 3,
        gnn_type: str = 'gcn',
        mask_ratio: float = 0.15,
        lambda_reg: float = 1.0,
        num_projections: int = 256,
        dropout: float = 0.1
    ):
        super().__init__()
        
        self.num_nodes = num_nodes
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.n_features = n_features
        self.embedding_dim = embedding_dim
        
        # Temporal encoder: Process each node's time series
        self.temporal_encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=dropout
        )
        
        # Graph-LeJEPA encoder for spatial relationships
        self.graph_lejepa = GraphLeJEPA(
            num_features=hidden_dim,  # Input from temporal encoder
            hidden_dim=hidden_dim,
            embedding_dim=embedding_dim,
            num_layers=num_gnn_layers,
            gnn_type=gnn_type,
            mask_ratio=mask_ratio,
            num_projections=num_projections,
            lambda_reg=lambda_reg,
            dropout=dropout
        )
        
        # Prediction head
        self.predictor = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, pred_len)
        )
        
        self.mask_ratio = mask_ratio
        
    def encode_temporal(self, x, extra=None):
        """
        Encode temporal patterns for each node.
        
        Args:
            x: [batch, nodes, seq_len] - main features
            extra: [batch, nodes, seq_len, n_aux] - auxiliary features (optional)
        
        Returns:
            node_features: [batch * nodes, hidden_dim]
        """
        batch_size, num_nodes, seq_len = x.shape
        
        # Combine main and auxiliary features if available
        if extra is not None and extra.numel() > 0:
            # x: [batch, nodes, seq_len] -> [batch, nodes, seq_len, 1]
            x_expanded = x.unsqueeze(-1)
            # extra: [batch, nodes, seq_len, n_aux]
            x_combined = torch.cat([x_expanded, extra], dim=-1)
        else:
            x_combined = x.unsqueeze(-1)
        
        # Reshape for LSTM: [batch * nodes, seq_len, n_features]
        x_flat = x_combined.view(batch_size * num_nodes, seq_len, -1)
        
        # Temporal encoding
        _, (h_n, _) = self.temporal_encoder(x_flat)
        
        # Use last hidden state: [batch * nodes, hidden_dim]
        temporal_features = h_n[-1]
        
        return temporal_features, batch_size
    
    def forward(self, x, extra=None, edge_index=None, pretrain=False):
        """
        Forward pass.
        
        Args:
            x: [batch, nodes, seq_len] or [nodes, seq_len]
            extra: Auxiliary features [batch, nodes, seq_len, n_aux]
            edge_index: [2, E] graph edges
            pretrain: If True, return JEPA losses for pre-training
        
        Returns:
            If pretrain: dict with pred_loss, reg_loss, total_loss
            Else: predictions [batch, nodes]
        """
        # Handle both batched and unbatched input
        if x.dim() == 2:
            x = x.unsqueeze(0)
            if extra is not None and extra.dim() == 3:
                extra = extra.unsqueeze(0)
        
        batch_size = x.shape[0]
        num_nodes = x.shape[1]
        
        # Temporal encoding
        temporal_features, _ = self.encode_temporal(x, extra)
        
        if edge_index is None:
            # If no graph provided, use fully connected
            src = torch.arange(num_nodes).repeat(num_nodes)
            dst = torch.arange(num_nodes).repeat_interleave(num_nodes)
            edge_index = torch.stack([src, dst]).to(x.device)
        
        # Create batch indices for graph batching
        batch_indices = torch.arange(batch_size).repeat_interleave(num_nodes).to(x.device)
        
        # Adjust edge_index for batched graphs
        edge_index_batched = []
        for b in range(batch_size):
            offset = b * num_nodes
            edge_index_batched.append(edge_index + offset)
        edge_index_batched = torch.cat(edge_index_batched, dim=1).to(x.device)
        
        if pretrain:
            # Pre-training mode: return JEPA losses
            jepa_output = self.graph_lejepa(
                temporal_features,
                edge_index_batched,
                batch_indices,
                mask_type='random'
            )
            return jepa_output
        else:
            # Inference mode: get embeddings and predict
            node_emb, _ = self.graph_lejepa.encode(
                temporal_features,
                edge_index_batched,
                batch_indices
            )
            
            # Predict for each node: [batch * nodes, pred_len]
            predictions = self.predictor(node_emb)
            
            # Reshape to [batch, nodes, pred_len]
            predictions = predictions.view(batch_size, num_nodes, -1)
            
            # Squeeze if pred_len == 1
            if self.pred_len == 1:
                predictions = predictions.squeeze(-1)
            
            return predictions

print("GraphLeJEPAPredictor defined successfully")

GraphLeJEPAPredictor defined successfully


## Initialize Graph-LeJEPA Model

In [18]:
# Initialize model
model = GraphLeJEPAPredictor(
    num_nodes=n_sites,
    seq_len=CONFIG['seq_len'],
    pred_len=CONFIG['pred_len'],
    n_features=n_features,
    hidden_dim=CONFIG['hidden_dim'],
    embedding_dim=CONFIG['embedding_dim'],
    num_gnn_layers=CONFIG['num_gnn_layers'],
    gnn_type=CONFIG['gnn_type'],
    mask_ratio=CONFIG['mask_ratio'],
    lambda_reg=CONFIG['lambda_reg'],
    num_projections=CONFIG['num_projections'],
    dropout=CONFIG['dropout']
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

Model initialized:
  Total parameters: 276,993
  Trainable parameters: 276,993


## Phase 1: Pre-training with JEPA Objective

Pre-train the model using the Graph-JEPA self-supervised objective:
- Mask subgraphs and predict their embeddings from context
- SIGReg regularization prevents embedding collapse

In [ ]:
def pretrain_graph_lejepa(model, train_loader, edge_index, epochs, lr, device):
    """
    Pre-train Graph-LeJEPA using self-supervised JEPA objective.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'pred_loss': [], 'reg_loss': [], 'total_loss': []}
    
    model.train()
    
    for epoch in tqdm(range(epochs), desc='Pre-training'):
        epoch_losses = {'pred_loss': 0, 'reg_loss': 0, 'total_loss': 0}
        num_batches = 0
        
        for feat, label, extra in train_loader:
            optimizer.zero_grad()
            
            # feat: [batch, nodes, seq_len]
            # extra: [batch, nodes, seq_len, n_aux] or empty
            if extra.numel() == 0:
                extra = None
            
            # Forward pass in pre-training mode
            output = model(feat, extra, edge_index, pretrain=True)
            
            loss = output['total_loss']
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_losses['pred_loss'] += output['pred_loss'].item()
            epoch_losses['reg_loss'] += output['reg_loss'].item()
            epoch_losses['total_loss'] += output['total_loss'].item()
            num_batches += 1
        
        scheduler.step()
        
        # Average losses
        for k in epoch_losses:
            epoch_losses[k] /= max(num_batches, 1)
            history[k].append(epoch_losses[k])
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}: pred_loss={epoch_losses['pred_loss']:.4f}, "
                  f"reg_loss={epoch_losses['reg_loss']:.4f}, total={epoch_losses['total_loss']:.4f}")
    
    return history

print("Starting pre-training...")
pretrain_history = pretrain_graph_lejepa(
    model,
    train_loader,
    edge_index,
    epochs=CONFIG['pretrain_epochs'],
    lr=CONFIG['pretrain_lr'],
    device=device
)
print("Pre-training completed!")

Starting pre-training...


Pre-training:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
# Plot pre-training losses
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(pretrain_history['pred_loss'])
axes[0].set_title('Prediction Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(pretrain_history['reg_loss'])
axes[1].set_title('SIGReg Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')

axes[2].plot(pretrain_history['total_loss'])
axes[2].set_title('Total Loss')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')

plt.tight_layout()
os.makedirs('../results/g_lejepa', exist_ok=True)
plt.savefig('../results/g_lejepa/pretrain_loss.png', dpi=150)
plt.show()

## Phase 2: Fine-tuning for Prediction

Fine-tune the pre-trained model for demand prediction using supervised learning.

In [ ]:
def finetune_graph_lejepa(model, train_loader, valid_loader, edge_index, epochs, lr, device, save_path):
    """
    Fine-tune Graph-LeJEPA for prediction task.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    criterion = nn.MSELoss()
    
    history = {'train_loss': [], 'valid_loss': []}
    best_valid_loss = float('inf')
    
    for epoch in tqdm(range(epochs), desc='Fine-tuning'):
        # Training
        model.train()
        train_loss = 0
        num_batches = 0
        
        for feat, label, extra in train_loader:
            optimizer.zero_grad()
            
            if extra.numel() == 0:
                extra = None
            
            # Forward pass in prediction mode
            predictions = model(feat, extra, edge_index, pretrain=False)
            
            # Ensure shapes match
            if predictions.dim() != label.dim():
                if predictions.dim() > label.dim():
                    predictions = predictions.squeeze(-1)
                else:
                    label = label.squeeze(-1)
            
            loss = criterion(predictions, label)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            num_batches += 1
        
        train_loss /= max(num_batches, 1)
        history['train_loss'].append(train_loss)
        
        # Validation
        model.eval()
        valid_loss = 0
        num_batches = 0
        
        with torch.no_grad():
            for feat, label, extra in valid_loader:
                if extra.numel() == 0:
                    extra = None
                
                predictions = model(feat, extra, edge_index, pretrain=False)
                
                if predictions.dim() != label.dim():
                    if predictions.dim() > label.dim():
                        predictions = predictions.squeeze(-1)
                    else:
                        label = label.squeeze(-1)
                
                loss = criterion(predictions, label)
                valid_loss += loss.item()
                num_batches += 1
        
        valid_loss /= max(num_batches, 1)
        history['valid_loss'].append(valid_loss)
        
        scheduler.step(valid_loss)
        
        # Save best model
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(save_path, 'best_model.pth'))
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}: train_loss={train_loss:.4f}, valid_loss={valid_loss:.4f}")
    
    # Load best model
    model.load_state_dict(torch.load(os.path.join(save_path, 'best_model.pth'), weights_only=True))
    
    return history

save_path = f'../results/g_lejepa/{CONFIG["city"]}'

print("Starting fine-tuning...")
finetune_history = finetune_graph_lejepa(
    model,
    train_loader,
    valid_loader,
    edge_index,
    epochs=CONFIG['finetune_epochs'],
    lr=CONFIG['finetune_lr'],
    device=device,
    save_path=save_path
)
print("Fine-tuning completed!")

In [ ]:
# Plot fine-tuning losses
plt.figure(figsize=(10, 4))
plt.plot(finetune_history['train_loss'], label='Train Loss')
plt.plot(finetune_history['valid_loss'], label='Valid Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Graph-LeJEPA Fine-tuning Loss')
plt.legend()
plt.savefig('../results/g_lejepa/finetune_loss.png', dpi=150)
plt.show()

## Evaluate Graph-LeJEPA on Test Set

In [ ]:
def evaluate_model(model, test_loader, edge_index, scaler, device):
    """
    Evaluate model on test set and compute metrics.
    """
    model.eval()
    predictions_list = []
    labels_list = []
    
    with torch.no_grad():
        for feat, label, extra in test_loader:
            if extra.numel() == 0:
                extra = None
            
            predictions = model(feat, extra, edge_index, pretrain=False)
            
            if predictions.dim() != label.dim():
                if predictions.dim() > label.dim():
                    predictions = predictions.squeeze(-1)
                else:
                    label = label.squeeze(-1)
            
            predictions_list.append(predictions.cpu().numpy())
            labels_list.append(label.cpu().numpy())
    
    # Concatenate
    pred_array = np.concatenate(predictions_list, axis=0)
    label_array = np.concatenate(labels_list, axis=0)
    
    # Inverse transform
    if scaler is not None:
        pred_array = scaler.inverse_transform(pred_array)
        label_array = scaler.inverse_transform(label_array)
    
    # Calculate metrics
    metrics = calculate_regression_metrics(label_array, pred_array)
    
    return metrics, pred_array, label_array

# Evaluate Graph-LeJEPA
g_lejepa_metrics, g_lejepa_preds, g_lejepa_labels = evaluate_model(
    model, test_loader, edge_index, ev_dataset.scaler, device
)

print("\n" + "="*60)
print("GRAPH-LeJEPA RESULTS")
print("="*60)
for metric_name, value in g_lejepa_metrics.items():
    print(f"  {metric_name}: {value:.4f}")

## Run Baseline Models for Comparison

In [ ]:
# Import baseline model components
from api.model.config import PredictionModel
from api.trainer.common import PredictionTrainer
from api.dataset.common import EVDataset

def run_baseline_model(model_name, config, device):
    """
    Run a baseline model and return metrics.
    """
    print(f"\nTraining {model_name}...")
    
    data_path = f'../data/{config["city"]}_remove_zero/'
    
    # Try to load dataset - handle column name issues
    try:
        # First, fix the sites.csv if needed
        sites_df = pd.read_csv(f'{data_path}sites.csv')
        if 'site_id' not in sites_df.columns and 'site' in sites_df.columns:
            sites_df = sites_df.rename(columns={'site': 'site_id'})
            sites_df.to_csv(f'{data_path}sites_fixed.csv', index=False)
        
        # Load dataset
        ev_ds = EVDataset(
            feature=config['feature'],
            auxiliary=config['auxiliary'],
            data_path=data_path,
            max_sites=config['max_sites'],
            selection_mode='top'
        )
        
        # Split data
        # Use time-based split similar to our custom approach
        n = len(ev_ds.feat)
        train_end = int(n * config['train_ratio'])
        valid_end = int(n * (config['train_ratio'] + config['valid_ratio']))
        
        ev_ds.scaler = StandardScaler()
        ev_ds.train_feat = ev_ds.scaler.fit_transform(ev_ds.feat[:train_end])
        ev_ds.valid_feat = ev_ds.scaler.transform(ev_ds.feat[train_end:valid_end])
        ev_ds.test_feat = ev_ds.scaler.transform(ev_ds.feat[valid_end:])
        
        if ev_ds.extra_feat is not None:
            ev_ds.train_extra_feat = ev_ds.extra_feat[:train_end]
            ev_ds.valid_extra_feat = ev_ds.extra_feat[train_end:valid_end]
            ev_ds.test_extra_feat = ev_ds.extra_feat[valid_end:]
        else:
            ev_ds.train_extra_feat = ev_ds.valid_extra_feat = ev_ds.test_extra_feat = None
        
        # Create loaders
        ev_ds.create_loaders(
            seq_l=config['seq_len'],
            pre_len=config['pred_len'],
            batch_size=config['batch_size'],
            device=device
        )
        
        n_features = 1 + (ev_ds.extra_feat.shape[-1] if ev_ds.extra_feat is not None else 0)
        
        # Initialize model
        pred_model = PredictionModel(
            num_node=ev_ds.feat.shape[1],
            n_fea=n_features,
            model_name=model_name,
            seq_l=config['seq_len'],
            pre_len=config['pred_len']
        )
        pred_model.model = pred_model.model.to(device)
        
        save_path = f'../results/baselines/{model_name}_{config["city"]}'
        
        # Initialize trainer
        trainer = PredictionTrainer(
            dataset=ev_ds,
            model=pred_model,
            seq_l=config['seq_len'],
            pre_len=config['pred_len'],
            is_train=True,
            save_path=save_path
        )
        
        # Train
        if model_name not in ['lo', 'ar', 'arima']:
            trainer.training(epoch=config['baseline_epochs'])
        
        # Test
        model_path = os.path.join(save_path, 'train.pth') if model_name not in ['lo', 'ar', 'arima'] else None
        trainer.test(model_path=model_path)
        
        # Load metrics
        with open(os.path.join(save_path, 'metrics.json'), 'r') as f:
            metrics = json.load(f)
        
        return metrics
        
    except Exception as e:
        print(f"  Error: {e}")
        return None

# Run baseline models
baseline_results = {}

for model_name in CONFIG['baseline_models']:
    metrics = run_baseline_model(model_name, CONFIG, device)
    if metrics is not None:
        baseline_results[model_name] = metrics
        print(f"  {model_name} - MAE: {metrics['MAE']:.4f}, RMSE: {metrics['RMSE']:.4f}, R2: {metrics['R²']:.4f}")
    else:
        baseline_results[model_name] = {'MAE': float('nan'), 'RMSE': float('nan'), 'R²': float('nan')}

print("\nBaseline evaluation completed!")

## Compare All Models

In [ ]:
# Compile all results
all_results = {'Graph-LeJEPA': g_lejepa_metrics}
all_results.update(baseline_results)

# Create comparison DataFrame
results_df = pd.DataFrame(all_results).T
results_df = results_df[['MAE', 'RMSE', 'MAPE', 'R²', 'EVS']]

print("\n" + "="*80)
print("MODEL COMPARISON - Shenzhen EV Charging Demand Prediction")
print("="*80)
print(results_df.round(4).to_string())

# Save results
os.makedirs('../results/g_lejepa', exist_ok=True)
results_df.to_csv('../results/g_lejepa/model_comparison.csv')
print("\nResults saved to results/g_lejepa/model_comparison.csv")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Filter out models with NaN results
valid_models = [m for m in all_results.keys() if not np.isnan(all_results[m].get('MAE', np.nan))]

# MAE comparison
ax1 = axes[0]
mae_values = [all_results[m]['MAE'] for m in valid_models]
colors = ['#2ecc71' if m == 'Graph-LeJEPA' else '#3498db' for m in valid_models]
bars = ax1.bar(valid_models, mae_values, color=colors)
ax1.set_title('Mean Absolute Error (MAE)', fontsize=12, fontweight='bold')
ax1.set_ylabel('MAE (lower is better)')
ax1.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, mae_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# RMSE comparison
ax2 = axes[1]
rmse_values = [all_results[m]['RMSE'] for m in valid_models]
bars = ax2.bar(valid_models, rmse_values, color=colors)
ax2.set_title('Root Mean Square Error (RMSE)', fontsize=12, fontweight='bold')
ax2.set_ylabel('RMSE (lower is better)')
ax2.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, rmse_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# R² comparison
ax3 = axes[2]
r2_values = [all_results[m]['R²'] for m in valid_models]
bars = ax3.bar(valid_models, r2_values, color=colors)
ax3.set_title('R² Score', fontsize=12, fontweight='bold')
ax3.set_ylabel('R² (higher is better)')
ax3.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, r2_values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../results/g_lejepa/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved to results/g_lejepa/model_comparison.png")

## Prediction Visualization

In [ ]:
# Visualize predictions for a few stations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Select 4 stations to visualize
stations_to_plot = [0, n_sites//4, n_sites//2, min(3*n_sites//4, n_sites-1)]
time_steps = min(200, len(g_lejepa_labels))

for idx, (ax, station) in enumerate(zip(axes.flat, stations_to_plot)):
    ax.plot(g_lejepa_labels[:time_steps, station], label='Actual', alpha=0.8)
    ax.plot(g_lejepa_preds[:time_steps, station], label='Predicted', alpha=0.8)
    ax.set_title(f'Station {station} - Charging Demand', fontsize=11)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Demand')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Graph-LeJEPA Predictions vs Actual Demand', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/g_lejepa/predictions_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("\n" + "="*80)
print("PHASE 4 COMPLETE: GRAPH-LeJEPA BENCHMARKING")
print("="*80)

# Filter valid results
valid_results = {k: v for k, v in all_results.items() if not np.isnan(v.get('MAE', np.nan))}

if valid_results:
    # Find best model
    best_model = min(valid_results.keys(), key=lambda m: valid_results[m]['MAE'])
    best_mae = valid_results[best_model]['MAE']
    
    print(f"\nBest Model: {best_model}")
    print(f"Best MAE: {best_mae:.4f}")
    
    # Graph-LeJEPA performance
    g_lejepa_mae = all_results['Graph-LeJEPA']['MAE']
    print(f"\nGraph-LeJEPA Performance:")
    print(f"  MAE: {g_lejepa_mae:.4f}")
    print(f"  RMSE: {all_results['Graph-LeJEPA']['RMSE']:.4f}")
    print(f"  R²: {all_results['Graph-LeJEPA']['R²']:.4f}")
    
    # Comparison with baselines
    print(f"\nComparison with Baselines:")
    for model_name in CONFIG['baseline_models']:
        if model_name in baseline_results and not np.isnan(baseline_results[model_name].get('MAE', np.nan)):
            baseline_mae = baseline_results[model_name]['MAE']
            if baseline_mae > 0:
                improvement = (baseline_mae - g_lejepa_mae) / baseline_mae * 100
                print(f"  vs {model_name}: {improvement:+.2f}% MAE {'improvement' if improvement > 0 else 'degradation'}")

print(f"\nResults saved to:")
print(f"  - results/g_lejepa/model_comparison.csv")
print(f"  - results/g_lejepa/model_comparison.png")
print(f"  - results/g_lejepa/predictions_visualization.png")
print(f"  - results/g_lejepa/pretrain_loss.png")
print(f"  - results/g_lejepa/finetune_loss.png")

print("\n" + "="*80)